# BIP-39 and BIP-32: Key Derivation by Foot

Companion to [BIP-39 and BIP-32: Understanding Key Derivation](https://weisser-zwerg.dev/posts/bip39-bip32-key-derivation/).

We start with 11-bit word-list indices, construct and validate recovery phrases, and derive the 64-byte BIP-39 seed using Python's standard library.
We then implement BIP-32 child derivation and follow Bitcoin and Ethereum paths, using `coincurve` only for the secp256k1 public-key primitive.
If you prefer the concise library approach, open the [bip_utils notebook](bip39-bip32-key-derivation-bip-utils.ipynb).

In [1]:
%load_ext watermark
%watermark -a 'cs224' -u -d -v -p bip_utils,notebook,nbclassic,watermark,coincurve

Author: cs224

Last updated: 2026-11-25

Python implementation: CPython
Python version       : 3.12.9
IPython version      : 9.19.1

bip_utils: 2.12.3
notebook : 7.6.5
nbclassic: 1.3.3
watermark: 2.6.0
coincurve: 21.0.0



In [2]:
from IPython.display import HTML, display

# Notebook 6 and nbclassic
# https://github.com/jupyter/nbclassic
display(HTML("<style>.container { width:70% !important; }</style>"))

# Notebook 7
display(HTML("<style>:root { --jp-notebook-max-width: 70% !important; }</style>"))

## Public Example Recovery Phrases

We use the public all-zero-entropy examples: `abandon` repeated 11 times followed by `about`, or 23 times followed by `art`.
These predictable phrases are for learning and must never hold funds.

Start with `ENTROPY_BITS = 128` and an empty `BIP39_PASSPHRASE`.
Set `ENTROPY_BITS = 256` and rerun the notebook from the beginning for the 24-word example.

In [3]:
MNEMONIC_12 = "abandon " * 11 + "about"
MNEMONIC_24 = "abandon " * 23 + "art"

ENTROPY_BITS = 128  # Use 256 for the 24-word example.
BIP39_PASSPHRASE = ""

print("12-word public example:", MNEMONIC_12)
print("24-word public example:", MNEMONIC_24)

12-word public example: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about
24-word public example: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon art


## Each Word Represents 11 Bits

[BIP-39](https://github.com/bitcoin/bips/blob/master/bip-0039.mediawiki) uses a defined list of 2,048 words.
Since `2048 = 2 ** 11`, each zero-based index fits into exactly 11 bits.
We read the local `english.txt` copy in its original order, then print a few indices and their binary representations.
The source and license of the list are recorded in the README.

| Index | 11-bit representation | Word |
| --- | --- | --- |
| 0 | `00000000000` | `abandon` |
| 1 | `00000000001` | `ability` |
| 3 | `00000000011` | `about` |
| 102 | `00001100110` | `art` |
| 2047 | `11111111111` | `zoo` |

In [4]:
from pathlib import Path

wordlist = Path("english.txt").read_text(encoding="utf-8").splitlines()
assert len(wordlist) == 2048
assert len(set(wordlist)) == 2048

for index in (0, 1, 3, 102, 2047):
    print(f"{index:4d}  {index:011b}  {wordlist[index]}")

   0  00000000000  abandon
   1  00000000001  ability
   3  00000000011  about
 102  00001100110  art
2047  11111111111  zoo


## Starting Bytes and the Checksum

For our reproducible examples, use all-zero bytes: 16 bytes for 128 bits, or 32 bytes for 256 bits.
Fresh wallet entropy instead needs cryptographically secure random bytes; the commented `secrets.token_bytes()` line shows that operation.
Hashing a predictable input does not make it unpredictable.

Append the first `entropy_bits // 32` bits of SHA-256 to the entropy.
The checksum is at most eight bits long, so we can read it from the first digest byte.
It adds error checking, not fresh randomness.

| Entropy bits | Checksum bits | Total bits | Words |
| --- | --- | --- | --- |
| 128 | 4 | 132 | 12 |
| 160 | 5 | 165 | 15 |
| 192 | 6 | 198 | 18 |
| 224 | 7 | 231 | 21 |
| 256 | 8 | 264 | 24 |

In [5]:
import hashlib
import secrets

entropy = bytes(ENTROPY_BITS // 8)
# For fresh random starting bytes, use:
# entropy = secrets.token_bytes(ENTROPY_BITS // 8)

entropy_bits = "".join(f"{byte:08b}" for byte in entropy)
checksum_length = len(entropy_bits) // 32
checksum_hash = hashlib.sha256(entropy).digest()
checksum_bits = f"{checksum_hash[0]:08b}"[:checksum_length]
bits = entropy_bits + checksum_bits

print("Entropy (hex):", entropy.hex())
print("Entropy bits:", len(entropy_bits))
print("Checksum bits:", checksum_bits)
print("Total bits:", len(bits))

Entropy (hex): 00000000000000000000000000000000
Entropy bits: 128
Checksum bits: 0011
Total bits: 132


## From Bits to Words

Split the combined bits into 11-bit groups, interpret each as an integer, and use that index to select a word.
The `:08b` and `:011b` formats preserve leading zeroes at the byte and word levels.

In [6]:
groups = [bits[offset:offset + 11] for offset in range(0, len(bits), 11)]
indices = [int(group, 2) for group in groups]
mnemonic = " ".join(wordlist[index] for index in indices)

for position, (group, index) in enumerate(zip(groups, indices), start=1):
    print(f"{position:2d}  {group}  {index:4d}  {wordlist[index]}")

print("\nRecovery phrase:", mnemonic)

 1  00000000000     0  abandon
 2  00000000000     0  abandon
 3  00000000000     0  abandon
 4  00000000000     0  abandon
 5  00000000000     0  abandon
 6  00000000000     0  abandon
 7  00000000000     0  abandon
 8  00000000000     0  abandon
 9  00000000000     0  abandon
10  00000000000     0  abandon
11  00000000000     0  abandon
12  00000000011     3  about

Recovery phrase: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about


### Why the Last Word Is about or art

For 128 zero bits, the last group is seven zero entropy bits plus checksum `0011`: `00000000011`, index 3, or `about`.
For 256 zero bits, it is three zero entropy bits plus checksum `01100110`: `00001100110`, index 102, or `art`.

The earlier groups are all zero and select `abandon`.
The last word contains both entropy and checksum bits; it is not a separate checksum word.
Change `ENTROPY_BITS` above and rerun from the beginning to see both cases.

## Recovering and Validating the Entropy

Reverse the word-list lookup, separate the checksum, and hash the recovered bytes to check it.
For valid phrase lengths, the checksum has `word_count // 3` bits.
The fixed byte length in `to_bytes()` retains leading zero bytes.

An unknown word, invalid count, or checksum mismatch is rejected.
Some changed phrases still have valid checksums; validity alone does not identify the intended wallet.

In [7]:
words = mnemonic.split()
if len(words) not in (12, 15, 18, 21, 24):
    raise ValueError("Invalid BIP-39 word count")

recovered_bits = "".join(f"{wordlist.index(word):011b}" for word in words)
received_checksum_length = len(words) // 3
recovered_entropy_bits = recovered_bits[:-received_checksum_length]
received_checksum = recovered_bits[-received_checksum_length:]

recovered_entropy = int(recovered_entropy_bits, 2).to_bytes(
    len(recovered_entropy_bits) // 8, "big"
)
expected_checksum = f"{hashlib.sha256(recovered_entropy).digest()[0]:08b}"[
    :received_checksum_length
]
if received_checksum != expected_checksum:
    raise ValueError("BIP-39 checksum does not match")

assert recovered_entropy == entropy
print("Recovered entropy:", recovered_entropy.hex())
print("Checksum matches:", received_checksum == expected_checksum)

Recovered entropy: 00000000000000000000000000000000
Checksum matches: True


## From Recovery Phrase to Seed

**PBKDF2 means Password-Based Key Derivation Function 2**, defined in **PKCS #5**.
It is a key derivation function (KDF): the recovery phrase is the starting secret, and the seed is the derived binary key material.
Python implements it in [`hashlib.pbkdf2_hmac(...)`](https://docs.python.org/3.12/library/hashlib.html#hashlib.pbkdf2_hmac).
BIP-39 uses HMAC-SHA512 with 2,048 iterations and a 64-byte output.

The `password` input is the selected recovery phrase, including the final word, with single spaces between the words.
The salt is `mnemonic` followed by `BIP39_PASSPHRASE`; with an empty passphrase it is just `mnemonic`.
Normalize both text inputs using NFKD, then encode them as UTF-8.
The next cell prints their exact byte representations before deriving the seed.

A salt is an additional derivation input; how it is combined with the other inputs depends on the KDF.
In [PBKDF2-HMAC](https://www.rfc-editor.org/rfc/rfc8018.html#section-5.2), the password serves as the HMAC key, while the salt followed by a block counter forms the first HMAC message.
The `"mnemonic" + BIP39_PASSPHRASE` expression builds the salt, which is passed separately to `hashlib.pbkdf2_hmac`.
For further reading, see [8.6 Key derivation with HKDF](https://livebook.manning.com/book/real-world-cryptography/chapter-8#102); HKDF is a separate KDF with its own construction.

The passphrase is a derivation input. Every different normalized passphrase gives another seed, without a wrong-password error.
Keep it available with the recovery phrase if you need to reconstruct this seed.
The 512-bit output does not increase the amount of randomness in the original input.

In [8]:
import unicodedata

password = unicodedata.normalize("NFKD", mnemonic).encode("utf-8")
salt = unicodedata.normalize(
    "NFKD", "mnemonic" + BIP39_PASSPHRASE
).encode("utf-8")

print(f"password = {password!r}")
print(f"salt = {salt!r}")

seed = hashlib.pbkdf2_hmac("sha512", password, salt, 2048, dklen=64)

print("Seed:", seed.hex())
print("Seed length (bytes):", len(seed))
print("Seed length (bits):", len(seed) * 8)

password = b'abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about'
salt = b'mnemonic'
Seed: 5eb00bbddcf069084889a8ab9155568165f5c453ccb85e70811aaed6f6da5fc19a5ac40b389cd370d086206dec8aa6c43daea6690f20ad3d8d48b2d2ce9e38e4
Seed length (bytes): 64
Seed length (bits): 512


## Compare with Published Test Vectors

The [reference vectors](https://github.com/trezor/python-mnemonic/blob/f5697ffc75547e0f2982b4f43dbd7f8810d09abd/vectors.json) use the optional passphrase `TREZOR`.
Use that same passphrase when comparing the two public mnemonics with the published seeds.
These checks are independent of the passphrase you selected for the main example and require no network access.

In [9]:
reference_vectors = [
    (
        MNEMONIC_12,
        "c55257c360c07c72029aebc1b53c05ed0362ada38ead3e3e9efa3708e5349553"
        "1f09a6987599d18264c1e1c92f2cf141630c7a3c4ab7c81b2f001698e7463b04",
    ),
    (
        MNEMONIC_24,
        "bda85446c68413707090a52022edd26a1c9462295029f2e60cd7c4f2bbd30971"
        "70af7a4d73245cafa9c3cca8d561a7c3de6f5d4a10be8ed2a5e608d68f92fcc8",
    ),
]

for reference_mnemonic, expected_seed in reference_vectors:
    reference_password = unicodedata.normalize("NFKD", reference_mnemonic).encode("utf-8")
    reference_salt = unicodedata.normalize("NFKD", "mnemonicTREZOR").encode("utf-8")
    reference_seed = hashlib.pbkdf2_hmac(
        "sha512", reference_password, reference_salt, 2048, dklen=64
    )
    assert reference_seed.hex() == expected_seed
    print(f"{len(reference_mnemonic.split())}-word published seed matches")

12-word published seed matches
24-word published seed matches


We now have the 64-byte BIP-39 `seed` and have verified the published examples.
The following cells use that seed as the input to BIP-32 key derivation.

## BIP-32: From Seed to Bitcoin and Ethereum Keys

[BIP-32](https://github.com/bitcoin/bips/blob/master/bip-0032.mediawiki) derives a tree of keys from one seed.
We start with a Bitcoin native SegWit key at `m/84'/0'/0'/0/0`, then follow an Ethereum path with the same implementation.
Bitcoin's legacy, nested SegWit, and native SegWit account conventions can coexist in one wallet:

| Bitcoin account type | Purpose | First receiving key path | Mainnet address prefix |
| --- | --- | --- | --- |
| Legacy P2PKH ([BIP-44](https://github.com/bitcoin/bips/blob/master/bip-0044.mediawiki)) | `44'` | `m/44'/0'/0'/0/0` | `1...` |
| Nested SegWit P2SH-P2WPKH ([BIP-49](https://github.com/bitcoin/bips/blob/master/bip-0049.mediawiki)) | `49'` | `m/49'/0'/0'/0/0` | `3...` |
| Native SegWit P2WPKH ([BIP-84](https://github.com/bitcoin/bips/blob/master/bip-0084.mediawiki)) | `84'` | `m/84'/0'/0'/0/0` | `bc1q...` |

### Reading the Path

The conventional structure is `m / purpose' / coin_type' / account' / change / address_index`.
Each slash represents one child derivation, starting from the master node.
For `m/84'/0'/0'/0/0`:

| Element | Role | Meaning in our Bitcoin example |
| --- | --- | --- |
| `m` | Master node | Starting point of the private key tree |
| `84'` | Purpose | BIP-84 native SegWit account convention |
| First `0'` | Coin type | Bitcoin mainnet |
| Second `0'` | Account | First account, numbered zero |
| First plain `0` | Change | External chain for receiving; `1` selects the internal change chain |
| Last `0` | Address index | First key on that chain; subsequent keys use `1`, `2`, and so on |

### Hardened Indices

An apostrophe marks a hardened index: add `2**31` to the displayed number.
Thus `84'` is `2147483732`, encoded as `80000054` in four-byte big-endian form.
`0'` becomes `80000000`, while normal `0` becomes `00000000`.

Hardened derivation uses parent private material.
Normal derivation supports deriving child public keys from a parent extended public key, enabling watch-only address generation.
Hardened account boundaries limit that public-only derivation and the parent-key exposure possible when a parent extended public key and a non-hardened child private key are both known.

In [10]:
BTC_PATH = "m/84'/0'/0'/0/0"
ETH_PATH = "m/44'/60'/0'/0/0"
HARDENED = 2 ** 31


def parse_path(path):
    parts = path.split("/")
    if parts[0] != "m":
        raise ValueError("Use an absolute private-key path starting with m")

    indices = []
    for part in parts[1:]:
        hardened = part.endswith("'")
        number = part[:-1] if hardened else part
        if not number.isascii() or not number.isdigit():
            raise ValueError(f"Invalid path element: {part!r}")
        index = int(number)
        if not 0 <= index < HARDENED:
            raise ValueError("Each path number must be between 0 and 2**31 - 1")
        indices.append(index + HARDENED if hardened else index)
    return indices


for label, index in zip(BTC_PATH.split("/")[1:], parse_path(BTC_PATH)):
    print(f"{label:>3}  integer={index:10d}  bytes={index.to_bytes(4, 'big').hex()}")

84'  integer=2147483732  bytes=80000054
 0'  integer=2147483648  bytes=80000000
 0'  integer=2147483648  bytes=80000000
  0  integer=         0  bytes=00000000
  0  integer=         0  bytes=00000000


### Master Private Key and Chain Code

Apply HMAC-SHA512 with the literal key `b"Bitcoin seed"` and our BIP-39 `seed` as the message.
Split its 64-byte result into a 32-byte private key and a 32-byte chain code.
The private scalar must lie between `1` and `n - 1`, where `n` is the secp256k1 generator's order.

The pair is an extended private key. Both halves are derived deterministically; no extra randomness is added.
The literal HMAC key is a public constant, while the seed carries the secret material.

In [11]:
import hmac

# Order of the secp256k1 generator point.
N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141


def master_from_seed(seed_bytes):
    if not 16 <= len(seed_bytes) <= 64:
        raise ValueError("BIP-32 seeds must contain 16 to 64 bytes")
    digest = hmac.new(b"Bitcoin seed", seed_bytes, hashlib.sha512).digest()
    private_key, chain_code = digest[:32], digest[32:]
    if not 1 <= int.from_bytes(private_key, "big") < N:
        raise ValueError("Invalid BIP-32 master key")
    return private_key, chain_code


master_private_key, master_chain_code = master_from_seed(seed)
print("HMAC key:", b"Bitcoin seed")
print("HMAC message (seed):", seed.hex())
print("Master private key (I_L):", master_private_key.hex())
print("Master chain code (I_R):", master_chain_code.hex())

HMAC key: b'Bitcoin seed'
HMAC message (seed): 5eb00bbddcf069084889a8ab9155568165f5c453ccb85e70811aaed6f6da5fc19a5ac40b389cd370d086206dec8aa6c43daea6690f20ad3d8d48b2d2ce9e38e4
Master private key (I_L): 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
Master chain code (I_R): 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e


### The Public-Key Primitive

Normal child derivation needs the parent's compressed public key: a 33-byte encoding of the secp256k1 point obtained by multiplying the generator by the private scalar.
The first byte is `02` or `03`, identifying the parity of the y coordinate; the remaining 32 bytes encode x.

We use the low-level [`coincurve`](https://github.com/ofek/coincurve) package, pinned to version `21.0.0`, for that operation.
The BIP-32 HMAC and scalar arithmetic below are implemented explicitly.

In [12]:
from coincurve import PublicKey


def compressed_public_key(private_key):
    return PublicKey.from_secret(private_key).format(compressed=True)


print("Master public key:", compressed_public_key(master_private_key).hex())

Master public key: 03d902f35f560e0470c63313c7369168d9d7df2d49bf295fd9fb7cb109ccee0494


### Deriving a Child Private Key

Use the parent chain code as the HMAC-SHA512 key.
Its 37-byte message has the following layout:

| Derivation | First 33 message bytes | Last four message bytes |
| --- | --- | --- |
| Hardened | `0x00` followed by the 32-byte parent private key | Encoded child index, including its hardened bit |
| Normal | The 33-byte compressed parent public key | Encoded child index |

Split the result into `I_L` and `I_R`.
The child private scalar is `(parent_scalar + integer(I_L)) % n`; `I_R` is the new chain code.
Encode the scalar as exactly 32 big-endian bytes, retaining leading zeroes.

Reject `integer(I_L) >= n` and a zero child scalar.
BIP-32 specifies trying the next index for an invalid result; this helper raises an error so that an updated path can be chosen and recorded explicitly.
`trace=True` prints the inputs and intermediate values.

In [13]:
def derive_child(parent_private_key, parent_chain_code, index, trace=False):
    if len(parent_private_key) != 32 or len(parent_chain_code) != 32:
        raise ValueError("Private keys and chain codes must each contain 32 bytes")
    parent_scalar = int.from_bytes(parent_private_key, "big")
    if not 1 <= parent_scalar < N:
        raise ValueError("Invalid parent private key")
    if not 0 <= index < 2 ** 32:
        raise ValueError("Child index must fit into four unsigned bytes")

    hardened = index >= HARDENED
    index_bytes = index.to_bytes(4, "big")
    if hardened:
        parent_data = b"\x00" + parent_private_key
    else:
        parent_data = compressed_public_key(parent_private_key)
    data = parent_data + index_bytes

    digest = hmac.new(parent_chain_code, data, hashlib.sha512).digest()
    left, child_chain_code = digest[:32], digest[32:]
    tweak = int.from_bytes(left, "big")
    if tweak >= N:
        raise ValueError("Invalid BIP-32 child; try the next index")
    child_scalar = (parent_scalar + tweak) % N
    if child_scalar == 0:
        raise ValueError("Invalid BIP-32 child; try the next index")
    child_private_key = child_scalar.to_bytes(32, "big")

    if trace:
        print("Hardened:", hardened)
        print("Child index:", index)
        print("Index bytes:", index_bytes.hex())
        print("HMAC key (parent chain code):", parent_chain_code.hex())
        print("HMAC message:", data.hex())
        print("I_L:", left.hex())
        print("I_R (child chain code):", child_chain_code.hex())
        print("Parent private key:", parent_private_key.hex())
        print("Child private key:", child_private_key.hex())

    return child_private_key, child_chain_code

### First Hardened Step: m to m/84'

The encoded index is `80000054`.
The message begins with a zero byte and the parent private key, followed by that four-byte index.

In [14]:
purpose_private_key, purpose_chain_code = derive_child(
    master_private_key, master_chain_code, HARDENED + 84, trace=True
)

Hardened: True
Child index: 2147483732
Index bytes: 80000054
HMAC key (parent chain code): 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e
HMAC message: 001837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf6780000054
I_L: db3254aa9427b8422ab16730d39b931c33e8342267beb5b952ea74cc3f856e35
I_R (child chain code): ca6a9276f135e836a2c3222f4901cc0f323adcbb39f746ef3c1ab2b7460209e0
Parent private key: 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
Child private key: f36a166922514e2e3c7f09e139b0aeff039cbf020606670b27554f7fe1a24d9c


### First Normal Step: the Receiving Chain

Two further hardened steps select Bitcoin and account zero.
Then index `0` selects the external chain at `m/84'/0'/0'/0`.
This message begins with the account's compressed public key.

In [15]:
coin_private_key, coin_chain_code = derive_child(
    purpose_private_key, purpose_chain_code, HARDENED + 0
)
account_private_key, account_chain_code = derive_child(
    coin_private_key, coin_chain_code, HARDENED + 0
)
external_private_key, external_chain_code = derive_child(
    account_private_key, account_chain_code, 0, trace=True
)

Hardened: False
Child index: 0
Index bytes: 00000000
HMAC key (parent chain code): 4a53a0ab21b9dc95869c4e92a161194e03c0ef3ff5014ac692f433c4765490fc
HMAC message: 02707a62fdacc26ea9b63b1c197906f56ee0180d0bcf1966e1a2da34f5f3a09a9b00000000
I_L: 9b04ef55756992a33846cef1e05aa1d2a72b8dae62aea873666893a103fc51fd
I_R (child chain code): 5fb17f32471143583da296b3af79da1a98f63db12389a0150a67610620925a38
Parent private key: e14f274d16ca0d91031b98b162618061d03930fa381af6d4caf44b01819ab6d4
Child private key: 7c5416a28c33a0343b6267a342bc2235bcb5e1c1eb80ff0c718a8015b560c790


### Walk the Complete Bitcoin Path

Repeat the same child operation at every level, retaining the resulting key and chain code for the next step.
Print each node and finish with the first receiving private key and its compressed public key.

In [16]:
def derive_path(root_private_key, root_chain_code, path, trace=False):
    private_key, chain_code = root_private_key, root_chain_code
    current_path = "m"
    nodes = [(current_path, private_key, chain_code)]

    for index in parse_path(path):
        private_key, chain_code = derive_child(private_key, chain_code, index)
        label = f"{index - HARDENED}'" if index >= HARDENED else str(index)
        current_path += "/" + label
        nodes.append((current_path, private_key, chain_code))

    if trace:
        for node_path, node_private_key, node_chain_code in nodes:
            print(node_path)
            print("  private key:", node_private_key.hex())
            print("  chain code:", node_chain_code.hex())
    return private_key, chain_code

In [17]:
btc_private_key, btc_chain_code = derive_path(
    master_private_key, master_chain_code, BTC_PATH, trace=True
)
btc_public_key = compressed_public_key(btc_private_key)
print("Bitcoin private key:", btc_private_key.hex())
print("Bitcoin public key:", btc_public_key.hex())

m
  private key: 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
  chain code: 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e
m/84'
  private key: f36a166922514e2e3c7f09e139b0aeff039cbf020606670b27554f7fe1a24d9c
  chain code: ca6a9276f135e836a2c3222f4901cc0f323adcbb39f746ef3c1ab2b7460209e0
m/84'/0'
  private key: fe87dbc6d34a7e2aca1d727d3922eaf7c21a7f7c95bb8858dd8429a8a69c14f4
  chain code: 6c2d070f9d27f0f5c7900f164b4037e09301826bb874202f2f1c8095413d36bb
m/84'/0'/0'
  private key: e14f274d16ca0d91031b98b162618061d03930fa381af6d4caf44b01819ab6d4
  chain code: 4a53a0ab21b9dc95869c4e92a161194e03c0ef3ff5014ac692f433c4765490fc
m/84'/0'/0'/0
  private key: 7c5416a28c33a0343b6267a342bc2235bcb5e1c1eb80ff0c718a8015b560c790
  chain code: 5fb17f32471143583da296b3af79da1a98f63db12389a0150a67610620925a38
m/84'/0'/0'/0/0
  private key: 4604b4b710fe91f584fff084e1a9159fe4f8408fff380596a604948474ce4fa3
  chain code: ab52cce8c3dc905c885ff8aa6d3374df3c27a38363acd3d62379

### Follow the Ethereum Path

For our Ethereum example, use `m/44'/60'/0'/0/0`.
BIP-44 supplies the hierarchy; [SLIP-44](https://github.com/satoshilabs/slips/blob/master/slip-0044.md) assigns Ethereum coin type `60`.
The underlying BIP-32 algorithm and the master node, including the literal `b"Bitcoin seed"` input, remain the same.
The different purpose and coin-type indices lead to another branch.

The compressed public key printed here is used by BIP-32; Ethereum address construction is a further operation.

In [18]:
eth_private_key, eth_chain_code = derive_path(
    master_private_key, master_chain_code, ETH_PATH, trace=True
)
eth_public_key = compressed_public_key(eth_private_key)
print("Ethereum private key:", eth_private_key.hex())
print("Ethereum compressed public key:", eth_public_key.hex())

m
  private key: 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
  chain code: 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e
m/44'
  private key: cd46049fb82a4fccbd967c5234e61152f4df87d01d1ef833420814c67ee4b7ea
  chain code: 45d3b0e8206db10a08d555317c7e245c5bbd12254ce968f3c79a959d4e6af98a
m/44'/60'
  private key: eea62fd3712ad7ff27f7145280d07f0041a51ef2f35432d6b488cb294cf79bef
  chain code: ff0603122d4dc5bf9b36606ef1b1b6e5a6c9e79fc6a4b08f7e871eee10457a71
m/44'/60'/0'
  private key: e9b91c428e679f857fbfa0203440eaeaabb0ca46d80e70b6aa6250696ac0dfc7
  chain code: d882718b7a42806803eeb17f7483f20620611adb88fc943c898dc5aba94c2819
m/44'/60'/0'/0
  private key: a29ac2cb17e31cdab42a8fe2d83f04f4b69c5e73bc8d3bf6b5dc96ac239b145a
  chain code: c879d136f02003dc804811e42390714ea06448aeaee158c62d65f12a421d988f
m/44'/60'/0'/0/0
  private key: 1ab42cc412b618bdea3a599e3c9bae199ebf030895b039e9db1e30dafb12b727
  chain code: 736094f4f24b67e838a4b3d23d31d229ca03e00c9bb99ce9

### Check Published BIP-32 and BIP-84 Vectors

The [BIP-32 vectors](https://github.com/bitcoin/bips/blob/master/bip-0032.mediawiki#test-vectors) below check a master node, a mixed hardened/normal path, and a leading-zero case.
The expected private keys and chain codes are the raw fields from the published extended private keys.

The [BIP-84 examples](https://github.com/bitcoin/bips/blob/master/bip-0084.mediawiki#test-vectors) check two receiving keys and one change key.
Their expected raw private bytes correspond to the published WIF values.
The reference mnemonic is always the public 12-word phrase with an empty passphrase, independent of the main example's selected inputs.

In [19]:
bip32_reference_nodes = [
    (
        "000102030405060708090a0b0c0d0e0f",
        "m",
        "e8f32e723decf4051aefac8e2c93c9c5b214313817cdb01a1494b917c8436b35",
        "873dff81c02f525623fd1fe5167eac3a55a049de3d314bb42ee227ffed37d508",
    ),
    (
        "000102030405060708090a0b0c0d0e0f",
        "m/0'/1/2'/2/1000000000",
        "471b76e389e528d6de6d816857e012c5455051cad6660850e58372a6c3e6e7c8",
        "c783e67b921d2beb8f6b389cc646d7263b4145701dadd2161548a8b078e65e9e",
    ),
    (
        "3ddd5602285899a946114506157c7997e5444528f3003f6134712147db19b678",
        "m/0'/1'",
        "3a2086edd7d9df86c3487a5905a1712a9aa664bce8cc268141e07549eaa8661d",
        "a48ee6674c5264a237703fd383bccd9fad4d9378ac98ab05e6e7029b06360c0d",
    ),
]

for seed_hex, path, expected_private, expected_chain in bip32_reference_nodes:
    reference_root = master_from_seed(bytes.fromhex(seed_hex))
    actual_private, actual_chain = derive_path(*reference_root, path)
    assert actual_private.hex() == expected_private
    assert actual_chain.hex() == expected_chain
    print("Published BIP-32 node matches:", path)

Published BIP-32 node matches: m
Published BIP-32 node matches: m/0'/1/2'/2/1000000000
Published BIP-32 node matches: m/0'/1'


In [20]:
# The published BIP-84 examples use the 12-word phrase and an empty passphrase.
reference_seed = hashlib.pbkdf2_hmac(
    "sha512",
    unicodedata.normalize("NFKD", MNEMONIC_12).encode("utf-8"),
    b"mnemonic",
    2048,
    dklen=64,
)
reference_root = master_from_seed(reference_seed)
bip84_reference_nodes = [
    (
        "m/84'/0'/0'/0/0",
        "4604b4b710fe91f584fff084e1a9159fe4f8408fff380596a604948474ce4fa3",
        "0330d54fd0dd420a6e5f8d3624f5f3482cae350f79d5f0753bf5beef9c2d91af3c",
    ),
    (
        "m/84'/0'/0'/0/1",
        "2fd0affa51529f940a358ec0c50de81267d0bf5158ca61887347676946362c5b",
        "03e775fd51f0dfb8cd865d9ff1cca2a158cf651fe997fdc9fee9c1d3b5e995ea77",
    ),
    (
        "m/84'/0'/0'/1/0",
        "3277578a56b721e4c9f071f1e24aa0f94c4ff72e7967fea03b134f605f07c8fd",
        "03025324888e429ab8e3dbaf1f7802648b9cd01e9b418485c5fa4c1b9b5700e1a6",
    ),
]

for path, expected_private, expected_public in bip84_reference_nodes:
    actual_private, _ = derive_path(*reference_root, path)
    assert actual_private.hex() == expected_private
    assert compressed_public_key(actual_private).hex() == expected_public
    print("Published BIP-84 key matches:", path)

Published BIP-84 key matches: m/84'/0'/0'/0/0
Published BIP-84 key matches: m/84'/0'/0'/0/1
Published BIP-84 key matches: m/84'/0'/0'/1/0


We have derived the individual Bitcoin and Ethereum private keys and their compressed public keys.
To reconstruct the same keys, retain the recovery phrase, any BIP-39 passphrase, and the exact derivation path.
Constructing a Bitcoin or Ethereum address from the public key is a further step.